
### 1. Imports & Config

In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')
 
# Paths
RAW_PATH = '../data/raw/HR_Employee_Attrition.csv'
PROCESSED_PATH = '../data/processed/hr_cleaned.csv'
print("Libraries loaded ✓")
 

Libraries loaded ✓


### 2. Load Data

In [9]:
df = pd.read_csv(RAW_PATH)
 
print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
df.head()
 
# Basic info
df.info()
 
# Check for missing values
missing = df.isnull().sum()
print("Missing values per column:")
print(missing[missing > 0] if missing.sum() > 0 else "No missing values ✓")
 

Task was destroyed but it is pending!
task: <Task pending name='Task-99' coro=<_async_in_context.<locals>.run_in_context_pre311() done, defined at C:\Users\baisa\AppData\Local\Programs\Python\Python310\lib\site-packages\ipykernel\utils.py:76> wait_for=<Task pending name='Task-100' coro=<_async_in_context.<locals>.preserve_context() running at C:\Users\baisa\AppData\Local\Programs\Python\Python310\lib\site-packages\ipykernel\utils.py:68> cb=[Task.task_wakeup()]> cb=[ZMQStream._run_callback.<locals>._log_error() at C:\Users\baisa\AppData\Local\Programs\Python\Python310\lib\site-packages\zmq\eventloop\zmqstream.py:563]>
Task was destroyed but it is pending!
task: <Task pending name='Task-100' coro=<_async_in_context.<locals>.preserve_context() running at C:\Users\baisa\AppData\Local\Programs\Python\Python310\lib\site-packages\ipykernel\utils.py:68> cb=[Task.task_wakeup()]>


Shape: (1470, 35)
Columns: ['Age', 'Attrition', 'BusinessTravel', 'DailyRate', 'Department', 'DistanceFromHome', 'Education', 'EducationField', 'EmployeeCount', 'EmployeeNumber', 'EnvironmentSatisfaction', 'Gender', 'HourlyRate', 'JobInvolvement', 'JobLevel', 'JobRole', 'JobSatisfaction', 'MaritalStatus', 'MonthlyIncome', 'MonthlyRate', 'NumCompaniesWorked', 'Over18', 'OverTime', 'PercentSalaryHike', 'PerformanceRating', 'RelationshipSatisfaction', 'StandardHours', 'StockOptionLevel', 'TotalWorkingYears', 'TrainingTimesLastYear', 'WorkLifeBalance', 'YearsAtCompany', 'YearsInCurrentRole', 'YearsSinceLastPromotion', 'YearsWithCurrManager']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1470 entries, 0 to 1469
Data columns (total 35 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   Age                       1470 non-null   int64 
 1   Attrition                 1470 non-null   object
 2   BusinessTravel            

### 3. Data Type Fixes & Encoding

In [10]:
# Binary encoding: Attrition Yes/No → 1/0
df['AttritionFlag'] = (df['Attrition'] == 'Yes').astype(int)
df['OverTimeFlag']  = (df['OverTime']  == 'Yes').astype(int)
 
# Encode satisfaction labels (1-4 scale stays numeric, just add labels)
satisfaction_map = {1: 'Low', 2: 'Medium', 3: 'High', 4: 'Very High'}
df['JobSatisfactionLabel'] = df['JobSatisfaction'].map(satisfaction_map)
 

### 4. Feature Engineering

In [11]:
# Salary band
def salary_band(income):
    if income < 3000:  return 'Low (<3K)'
    elif income < 6000: return 'Mid (3K-6K)'
    elif income < 10000: return 'High (6K-10K)'
    else: return 'Very High (10K+)'
 
df['SalaryBand'] = df['MonthlyIncome'].apply(salary_band)
 
# Tenure group (KEY FEATURE — tenure < 2 years = 3x attrition risk)
def tenure_group(years):
    if years < 2:   return '< 2 years'
    elif years < 5:  return '2-5 years'
    elif years < 10: return '5-10 years'
    else:            return '10+ years'
 
df['TenureGroup'] = df['YearsAtCompany'].apply(tenure_group)
 
# Age group
def age_group(age):
    if age < 25:   return '18-24'
    elif age < 35: return '25-34'
    elif age < 45: return '35-44'
    elif age < 55: return '45-54'
    else:          return '55+'
 
df['AgeGroup'] = df['Age'].apply(age_group)
 
# Risk score: composite (higher = more at risk)
# Based on: low tenure + low income + overtime + low satisfaction
df['RiskScore'] = (
    (df['YearsAtCompany'] < 2).astype(int) * 3 +
    (df['MonthlyIncome'] < 3000).astype(int) * 2 +
    (df['OverTimeFlag']) * 2 +
    (df['JobSatisfaction'] <= 2).astype(int) * 2 +
    (df['WorkLifeBalance'] <= 2).astype(int) * 1
)
 
print("Features engineered:")
print(df[['SalaryBand', 'TenureGroup', 'AgeGroup', 'RiskScore']].head(10))


Features engineered:
      SalaryBand TenureGroup AgeGroup  RiskScore
0    Mid (3K-6K)  5-10 years    35-44          3
1    Mid (3K-6K)   10+ years    45-54          2
2      Low (<3K)   < 2 years    35-44          7
3      Low (<3K)  5-10 years    25-34          4
4    Mid (3K-6K)   2-5 years    25-34          2
5    Mid (3K-6K)  5-10 years    25-34          1
6      Low (<3K)   < 2 years      55+         10
7      Low (<3K)   < 2 years    25-34          5
8  High (6K-10K)  5-10 years    35-44          0
9    Mid (3K-6K)  5-10 years    35-44          1


### 5. Drop Constant / Redundant Columns

In [12]:
# These columns have only one value in the IBM dataset
cols_to_drop = ['EmployeeCount', 'Over18', 'StandardHours']
df.drop(columns=cols_to_drop, inplace=True)
print(f"Dropped: {cols_to_drop}")
print(f"Final shape: {df.shape}")

Dropped: ['EmployeeCount', 'Over18', 'StandardHours']
Final shape: (1470, 39)



### 6. Validation Checks

In [13]:

print("=== Attrition Distribution ===")
print(df['Attrition'].value_counts())
print(f"\nAttrition Rate: {df['AttritionFlag'].mean()*100:.1f}%")
 
print("\n=== Salary Band Distribution ===")
print(df['SalaryBand'].value_counts())
 
print("\n=== Tenure Group Distribution ===")
print(df['TenureGroup'].value_counts())
 
print("\n=== Dtypes check ===")
print(df.dtypes)
 

=== Attrition Distribution ===
Attrition
No     1233
Yes     237
Name: count, dtype: int64

Attrition Rate: 16.1%

=== Salary Band Distribution ===
SalaryBand
Mid (3K-6K)         519
Low (<3K)           395
Very High (10K+)    281
High (6K-10K)       275
Name: count, dtype: int64

=== Tenure Group Distribution ===
TenureGroup
5-10 years    524
10+ years     366
2-5 years     365
< 2 years     215
Name: count, dtype: int64

=== Dtypes check ===
Age                          int64
Attrition                   object
BusinessTravel              object
DailyRate                    int64
Department                  object
DistanceFromHome             int64
Education                    int64
EducationField              object
EmployeeNumber               int64
EnvironmentSatisfaction      int64
Gender                      object
HourlyRate                   int64
JobInvolvement               int64
JobLevel                     int64
JobRole                     object
JobSatisfaction            

### 7. Export Processed Dataset

In [14]:
os.makedirs('../data/processed', exist_ok=True)
df.to_csv(PROCESSED_PATH, index=False)
print(f"Cleaned dataset saved to: {PROCESSED_PATH}")
print(f"Final shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

Cleaned dataset saved to: ../data/processed/hr_cleaned.csv
Final shape: (1470, 39)
Columns: ['Age', 'Attrition', 'BusinessTravel', 'DailyRate', 'Department', 'DistanceFromHome', 'Education', 'EducationField', 'EmployeeNumber', 'EnvironmentSatisfaction', 'Gender', 'HourlyRate', 'JobInvolvement', 'JobLevel', 'JobRole', 'JobSatisfaction', 'MaritalStatus', 'MonthlyIncome', 'MonthlyRate', 'NumCompaniesWorked', 'OverTime', 'PercentSalaryHike', 'PerformanceRating', 'RelationshipSatisfaction', 'StockOptionLevel', 'TotalWorkingYears', 'TrainingTimesLastYear', 'WorkLifeBalance', 'YearsAtCompany', 'YearsInCurrentRole', 'YearsSinceLastPromotion', 'YearsWithCurrManager', 'AttritionFlag', 'OverTimeFlag', 'JobSatisfactionLabel', 'SalaryBand', 'TenureGroup', 'AgeGroup', 'RiskScore']
